# step C — pool-500 (RQ2 인과, KV 치환, 500 이름 커버)

**대응 RQ:** RQ2 인과 — stepB는 'L25에서 위반 표기를 더 본다'를 **관측**했다. stepC는 그 표현을 준수 값으로 **바꿔서 준수 선호가 회복되는지** = 형태 신호가 표기 결정의 **원인**인지 본다.

**무엇을 하나** — 텍스트는 그대로 두고(`format_matrix`가 화면엔 그대로), 위반 이름이 **L25에서 갖는 KV를 준수 값으로 치환**(내부 표현 수술)한 뒤 준수 선호도 회복을 잰다.

**세 상태의 준수 선호 점수 `S = logP(준수 후보) − logP(위반 후보)`:** clean(천장) / baseline(위반, 실패) / intervened(위반이지만 L25만 준수 치환).
**회복률 = (S_int − S_base) / (S_clean − S_base).**

**donor 3종:** `compliant`(같은 이름 준수판=주효과) / `unrelated_camel`(무관 준수형=형태 통제) / `unrelated_snake`(무관 위반형=음성 통제).

**파일럿 대비 (POOL 확장):** 이름 창고 80→**504(50×50)**. 파일럿은 seed로 12개 랜덤 표집이었으나, scaleup은 **블록**으로 500+개 위반 이름을 개입 대상으로 덮는다. seed는 위치·donor 변주.

**코드 동일 보장(재현성):** 개입·측정은 파일럿과 **똑같은 `run(...)`**. 단 POOL은 표집이라 파일럿(80풀)과 글자 그대로 재현되진 않음 — 새 500 샘플(파일럿은 `results/stepC/` 불변 보존).

설계 문서: `docs/stepC/scaleup-500.md` (파일럿: `docs/stepC/plan.md`). 공유 POOL 코드: `stepB/pool-500`.

> **메모리(T4):** 개입은 output_attentions 안 씀(KV 캐시 편집), 짧은 프롬프트라 가볍다. **eager 불필요.**
> **규모:** 3 donor × 42 block × 1 seed = **126조건**(§3 stepC와 일치). **재개 가능.** 위치·donor 강건성 원하면 `SEEDS=[0,1,2]`.
> **검증:** 요약에서 S_clean > S_base(위반이 실제로 준수 선호를 떨어뜨림)를 sanity check.

In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepC/pool-500
!git checkout stepC/pool-500
!git pull --quiet origin stepC/pool-500
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — donor 3종 × 블록(500 이름 커버). 개입 L25 KV, 선행 전부 위반(n=0).
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)
from harness.tasks import NAME_PAIR_POOL

MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
DONORS = ['compliant', 'unrelated_camel', 'unrelated_snake']
LAYER = 25
N_FUNCTIONS = 12
N_BLOCKS = len(NAME_PAIR_POOL) // N_FUNCTIONS   # 504//12 = 42 블록(500+ 위반 이름 커버)
BLOCKS = list(range(N_BLOCKS))
SEEDS = [0]                                     # 블록이 다양성 제공. 위치·donor 강건성 원하면 [0,1,2]

def make(donor, block, s):
    return Condition(
        model=MODEL,
        preceding=PrecedingCode(n_compliant=0, n_functions=N_FUNCTIONS,
                                composition=Composition.POOL, pool_block=block),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL),
        intervention=Intervention(kind=InterventionKind.KEY_VALUE, layers=[LAYER], donor=donor),
        seed=s,
    )

conditions = [make(d, b, s) for d in DONORS for b in BLOCKS for s in SEEDS]

PREDICTION = ('compliant 치환 시 준수 선호 회복(≈A2 79%). unrelated_camel(무관 준수형)도 '
              '회복하면 신호는 내용 아닌 형태. unrelated_snake(위반형)는 회복 안 됨. '
              '500개 위반 이름(블록)에서 효과가 특정 단어에 묶이지 않음.')
print(f'풀 {len(NAME_PAIR_POOL)}개 이름, {N_BLOCKS}블록')
print(f'{len(conditions)} 조건 = {len(DONORS)} donor x {len(BLOCKS)} block x {len(SEEDS)} seed  @L{LAYER}')

In [ ]:
# 실행 — 조건별 개입 + 즉시 저장(재개) + 중간 donor별 누적 회복률. 로직은 harness가 수행(파일럿과 동일).
from collections import defaultdict
from harness import run, ResultRecord, save_result, result_path
from harness.results import load_result
from harness.model import load_model

STEP = 'stepC_scaleup500'
handle = load_model(MODEL)   # 개입은 eager 불필요
print('layers:', handle.num_layers, '| GQA:', handle.gqa_info())

acc = defaultdict(list)
new = skipped = 0
for i, c in enumerate(conditions, 1):
    p = result_path(c, step=STEP)
    if p.exists():
        rec = load_result(p); skipped += 1
    else:
        out = run(c, handle=handle)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ2', prediction=PREDICTION))
        rec = load_result(p); new += 1
    r = rec.metrics.extra['recovery']; d = rec.condition.intervention.donor
    if r is not None:
        acc[d].append(r)
    if i % 15 == 0 or i == len(conditions):
        print(f'[{i}/{len(conditions)}] 새 {new} / 건너뜀 {skipped}')
        for d in DONORS:
            if acc[d]:
                print(f'    donor={d:<16} 회복률 평균 {sum(acc[d])/len(acc[d]):+.3f} (n={len(acc[d])}, 이름 {len(acc[d])*12}개)')
print(f'완료: 새로 {new}, 건너뜀 {skipped}, 총 {len(conditions)}')

In [ ]:
# 결과 로드 — results/stepC_scaleup500/
from harness import result_path
from harness.results import load_result

records = [load_result(result_path(c, step=STEP)) for c in conditions]
print('로드:', len(records), '건 -> results/'+STEP+'/')

In [ ]:
# 요약 — donor별 회복률 + 세 상태 S. sanity: S_clean > S_base. 500 이름 집계.
import pandas as pd, matplotlib.pyplot as plt

rows = [{'donor': r.condition.intervention.donor,
         'S_clean': r.metrics.extra['S_clean'], 'S_base': r.metrics.extra['S_base'],
         'S_int': r.metrics.extra['S_int'], 'recovery': r.metrics.extra['recovery'],
         'n_sub': r.metrics.extra['n_substituted_tokens']} for r in records]
df = pd.DataFrame(rows)

print('=== sanity: 위반이 준수 선호를 떨어뜨리는가 (S_clean > S_base?) ===')
print(df.groupby('donor')[['S_clean','S_base','S_int']].mean().round(3))
print()
print('=== donor별 회복률 (이름 500 집계) ===')
print(df.groupby('donor')['recovery'].agg(['mean','std','count']).round(3))

g = df.groupby('donor')['recovery'].mean().reindex(DONORS)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
colors = ['#2E7D52', '#2563C9', '#C6771A']
ax[0].bar(range(len(g)), g.values, color=colors)
ax[0].axhline(0, color='#888', lw=.8); ax[0].axhline(1, color='#888', ls='--', lw=.8)
ax[0].set_xticks(range(len(g))); ax[0].set_xticklabels(g.index, rotation=12, fontsize=8)
ax[0].set_ylabel('recovery rate'); ax[0].set_title('Recovery by donor (1.0=clean, scaleup-500)')
sub = df[df.donor=='compliant'][['S_clean','S_base','S_int']].mean()
ax[1].bar(['clean','baseline','intervened'], sub.values, color=['#2E7D52','#B0392B','#2563C9'])
ax[1].set_ylabel('S (compliance preference)'); ax[1].set_title('Three states (donor=compliant)')
plt.tight_layout(); plt.show()

In [ ]:
# 결과 다운로드 — results/stepC_scaleup500 을 zip으로 묶어 내려받는다
import shutil
shutil.make_archive('stepC_scaleup500_results', 'zip', 'results/'+STEP)
try:
    from google.colab import files
    files.download('stepC_scaleup500_results.zip')
except Exception as e:
    print('Colab 아님(수동 다운로드): stepC_scaleup500_results.zip', e)